### MQR (Multi-Query Retriever)

In [2]:
pip install langchain-classic

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_classic.retrievers import MultiQueryRetriever

load_dotenv()

True

In [4]:
# Relevant health & wellness documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [5]:
# Initialize Google Gemini Embeddings
embedding_model = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-2")

# create FAISS vector store
vectorstore = FAISS.from_documents(documents=all_docs, embedding=embedding_model)

# create retrievers
similarity_retriever = vectorstore.as_retriever(search_type='similarity', search_kwargs={"k": 5})

#### Create Multiquery retriever and will comapre it with simple similarity retriver

In [9]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0
)

multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k":5}),
    llm=llm
)

In [10]:
# Query
query = "How to improve energy levels and maintain balance?"

In [11]:
# Retriever results
similarity_results = similarity_retriever.invoke(query)
multiquery_results = multiquery_retriever.invoke(query)

c:\Users\User\Desktop\GenAI\LangChain\venv\Lib\site-packages\langchain_google_genai\chat_models.py:3237: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


In [13]:
# printing similarity search results
for i,docs in enumerate(similarity_results):
    print(f"--- Result {1+i} ---")
    print(docs.page_content)

--- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.
--- Result 2 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.
--- Result 3 ---
Consuming leafy greens and fruits helps detox the body and improve longevity.
--- Result 4 ---
Deep sleep is crucial for cellular repair and emotional regulation.
--- Result 5 ---
Regular walking boosts heart health and can reduce symptoms of depression.


In [14]:
# printing MultiQuery retriever results
for i,docs in enumerate(multiquery_results):
    print(f"--- Result {i+1} ---")
    print(docs.page_content)

--- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.
--- Result 2 ---
Consuming leafy greens and fruits helps detox the body and improve longevity.
--- Result 3 ---
Regular walking boosts heart health and can reduce symptoms of depression.
--- Result 4 ---
Deep sleep is crucial for cellular repair and emotional regulation.
--- Result 5 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.
